In [1]:
#### 测试MCTS性能速度

import time
# 确保你已经正确导入了所有需要的类
from OthelloGame import OthelloGame
from OthelloPlayers import PureMCTSPlayer, RandomPlayer

# 1. 初始化游戏和玩家
game = OthelloGame(8)
mcts_brain = PureMCTSPlayer(game, num_simulations=100).play
random_brain = RandomPlayer(game).play

# 2. 制造一个“中盘”棋盘
# 开局的模拟路径太长（要走 60 步才结束），残局的路径太短（走几步就结束）
# 为了测出最真实的平均水平，我们让随机玩家先走 20 步，进入典型的中盘！
test_board = game.getInitBoard()
curPlayer = 1
for _ in range(20):
    if game.getGameEnded(test_board, curPlayer) != 0:
        break
    action = random_brain(test_board) if curPlayer == 1 else random_brain(game.getCanonicalForm(test_board, curPlayer))
    test_board, curPlayer = game.getNextState(test_board, curPlayer, action)

# 记得把棋盘视角转换回 1 号，传给 MCTS
if curPlayer == -1:
    test_board = game.getCanonicalForm(test_board, curPlayer)

print(f"=== 当前测试棋盘 (已走20步的复杂中盘) ===")
print(test_board)

# 3. 开始测速
print("\n=== 开始测速: Pure MCTS (100 次模拟) ===")
start_time = time.time()
action = mcts_brain(test_board)
end_time = time.time()

print(f" MCTS 决定走动作: {action} (行 {action//8}, 列 {action%8})")
print(f" MCTS (100次模拟) 思考这步棋耗时: {end_time - start_time:.4f} 秒")

=== 当前测试棋盘 (已走20步的复杂中盘) ===
[[ 0  0  0  0  0  0  0  0]
 [ 0  0  0 -1  0  0  0  0]
 [ 0  0 -1 -1  1  0  0  0]
 [ 0  0  1 -1  1 -1  0  0]
 [ 0  0  0 -1 -1 -1 -1  0]
 [ 0  0  1 -1 -1 -1 -1  0]
 [ 0  0 -1 -1 -1  0 -1  0]
 [ 0 -1  0 -1  1  0  0  0]]

=== 开始测速: Pure MCTS (100 次模拟) ===
✅ MCTS 决定走动作: 34 (行 4, 列 2)
⏱️ MCTS (100次模拟) 思考这步棋耗时: 3.9616 秒


In [7]:
import time
from OthelloPlayers import MinimaxPlayer

# 假设我们在测试 Depth 4
game = OthelloGame(8)
m4_brain = MinimaxPlayer(game, search_depth=8).play

# 弄一个初始偏中盘的复杂棋盘，不要用开局，开局选择少，看不出耗时
test_board = game.getInitBoard()
# (你可以手动让它走几步进入中盘)

print("=== 开始测速 ===")
start_time = time.time()
action = m4_brain(test_board)
end_time = time.time()

print(f"Depth=8 思考这步棋耗时: {end_time - start_time:.4f} 秒")

=== 开始测速 ===
Depth=8 思考这步棋耗时: 4.0376 秒


In [1]:
import sys
import os
sys.path.append(os.getcwd()) 

from OthelloGame import OthelloGame
# 导入你刚刚“原地升级”为限时版的玩家类
from OthelloPlayers import RandomPlayer, MinimaxPlayer, PureMCTSPlayer
from Arena import Arena 

def setup_drop_in_test(empty_spots_target, time_limit=4.5, num_games=4):
    """
    定点切入测试：先让盲人随机走棋，直到棋盘只剩下 empty_spots_target 个空位，
    然后再让限时版 MCTS 和 Minimax 接手剩下的残局，看谁更强！
    """
    game = OthelloGame(8)
    
    print(f"\n========================================================")
    print(f"🚀 开始定点切入测试 | 目标阶段: 剩余 {empty_spots_target} 个空位")
    print(f"⏱️ 统一限时: {time_limit} 秒/步 | 测试局数: {num_games} 局")
    print(f"========================================================")

    # 初始化限时版玩家 (设定每次思考最多 4.5 秒)
    mcts_player = PureMCTSPlayer(game, time_limit=time_limit).play
    minimax_player = MinimaxPlayer(game, time_limit=time_limit).play
    
    mcts_wins = 0
    minimax_wins = 0
    draws = 0
    
    for i in range(num_games):
        # 1. 制造指定残局 (随机乱走)
        board = game.getInitBoard()
        cur_player = 1
        random_brain = RandomPlayer(game).play
        
        # 计算需要瞎走多少步才能达到目标的空位数量
        steps_to_play = 60 - empty_spots_target
        
        for _ in range(steps_to_play):
            if game.getGameEnded(board, cur_player) != 0:
                break 
            action = random_brain(board) if cur_player == 1 else random_brain(game.getCanonicalForm(board, cur_player))
            board, cur_player = game.getNextState(board, cur_player, action)
            
        print(f"\n--- 第 {i+1}/{num_games} 局开始 --- 棋盘空位: {empty_spots_target}")
        
        # 2. 决定这局谁执黑谁执白 (交替进行，保证公平)
        if i % 2 == 0:
            p1, p2 = mcts_player, minimax_player
            p1_name, p2_name = "MCTS", "Minimax"
        else:
            p1, p2 = minimax_player, mcts_player
            p1_name, p2_name = "Minimax", "MCTS"
            
        # 3. 双神接管对战
        while game.getGameEnded(board, cur_player) == 0:
            if cur_player == 1:
                action = p1(board)
            else:
                action = p2(game.getCanonicalForm(board, cur_player))
            board, cur_player = game.getNextState(board, cur_player, action)
            
        # 4. 统计结果
        result = cur_player * game.getGameEnded(board, cur_player)
        winner = p1_name if (result == 1 and cur_player == 1) or (result == -1 and cur_player == -1) else p2_name
        if result == 0:
            winner = "Draw"
            
        if winner == "MCTS":
            mcts_wins += 1
        elif winner == "Minimax":
            minimax_wins += 1
        else:
            draws += 1
            
        print(f"🏁 局 {i+1} 结束 | 赢家: {winner}")
        
    print(f"\n🏆 [剩余 {empty_spots_target} 空位] 最终比分 -> MCTS: {mcts_wins} | Minimax: {minimax_wins} | 平局: {draws}")

# ==========================================
# 开始执行实验 A：极度残局 (只剩 14 步)
# ==========================================
if __name__ == "__main__":
    setup_drop_in_test(empty_spots_target=14, time_limit=4.5, num_games=4)


🚀 开始定点切入测试 | 目标阶段: 剩余 14 个空位
⏱️ 统一限时: 4.5 秒/步 | 测试局数: 4 局

--- 第 1/4 局开始 --- 棋盘空位: 14
🏁 局 1 结束 | 赢家: Minimax

--- 第 2/4 局开始 --- 棋盘空位: 14
🏁 局 2 结束 | 赢家: MCTS

--- 第 3/4 局开始 --- 棋盘空位: 14
🏁 局 3 结束 | 赢家: Minimax

--- 第 4/4 局开始 --- 棋盘空位: 14
🏁 局 4 结束 | 赢家: Minimax

🏆 [剩余 14 空位] 最终比分 -> MCTS: 1 | Minimax: 3 | 平局: 0


In [ ]:
import sys
import os
import matplotlib.pyplot as plt

sys.path.append(os.getcwd()) 

from OthelloGame import OthelloGame
from OthelloPlayers import RandomPlayer, MinimaxPlayer, PureMCTSPlayer
from Arena import Arena 

#time limit: 4.5
def run_comprehensive_benchmark(time_limit=4.5, num_games_per_stage=40):

    game = OthelloGame(8)
    # Testing will be conducted in phases
    test_stages = [15, 20, 25, 30, 35, 40, 45, 50, 55, 60]
    
    # Record the success rate at each stage.
    mcts_win_rates = []
    minimax_win_rates = []
    
    mcts_player = PureMCTSPlayer(game, time_limit=time_limit).play
    minimax_player = MinimaxPlayer(game, time_limit=time_limit).play
    
    print(f" Start test(have {len(test_stages)} stages to be tested, each stages has {num_games_per_stage} games) ")
    
    for empty_spots in test_stages:
        print(f" Testing phase: There are still {empty_spots} empty positions remaining.")
        
        mcts_wins = 0
        minimax_wins = 0
        draws = 0
        
        for i in range(num_games_per_stage):
            board = game.getInitBoard()
            cur_player = 1
            random_brain = RandomPlayer(game).play
            
            # Creating a mess
            steps_to_play = 60 - empty_spots
            for _ in range(steps_to_play):
                if game.getGameEnded(board, cur_player) != 0:
                    break 
                action = random_brain(board) if cur_player == 1 else random_brain(game.getCanonicalForm(board, cur_player))
                board, cur_player = game.getNextState(board, cur_player, action)
                
            # Replacement first mover
            if i % 2 == 0:
                p1, p2 = mcts_player, minimax_player
                p1_name, p2_name = "MCTS", "Minimax"
            else:
                p1, p2 = minimax_player, mcts_player
                p1_name, p2_name = "Minimax", "MCTS"
                
            while game.getGameEnded(board, cur_player) == 0:
                if cur_player == 1:
                    action = p1(board)
                else:
                    action = p2(game.getCanonicalForm(board, cur_player))
                board, cur_player = game.getNextState(board, cur_player, action)
                
            result = cur_player * game.getGameEnded(board, cur_player)
            winner = p1_name if (result == 1 and cur_player == 1) or (result == -1 and cur_player == -1) else p2_name
            if result == 0:
                winner = "Draw"
                
            if winner == "MCTS":
                mcts_wins += 1
            elif winner == "Minimax":
                minimax_wins += 1
            else:
                draws += 1
                
            print(f"The game {i+1}/{num_games_per_stage} ends | Winner: {winner}")
            
        # Calculate win rate
        mcts_win_rate = (mcts_wins / num_games_per_stage) * 100
        minimax_win_rate = (minimax_wins / num_games_per_stage) * 100
        
        mcts_win_rates.append(mcts_win_rate)
        minimax_win_rates.append(minimax_win_rate)
        
        print(f" [Remaining {empty_spots} steps] MCTS winning rate: {mcts_win_rate:.1f}% | Minimax winning rate: {minimax_win_rate:.1f}%")
        
    return test_stages, mcts_win_rates, minimax_win_rates


def plot_results(stages, mcts_rates, minimax_rates):

    plt.figure(figsize=(10, 6))
    
    # Drawing a line chart
    plt.plot(stages, mcts_rates, marker='o', linewidth=3, markersize=10, label='MCTS', color='#ff7f0e')
    plt.plot(stages, minimax_rates, marker='s', linewidth=3, markersize=10, label='Minimax (Alpha-Beta)', color='#1f77b4')
    
    # Set the format of the chart
    plt.title('Performance Comparison: MCTS vs Minimax at Different Game Stages\n(Time Limit: 4.5s / move)', fontsize=16, pad=15)
    plt.xlabel('Remaining Empty Spots (Game Complexity)', fontsize=14)
    plt.ylabel('Win Rate (%)', fontsize=14)
    
    plt.xlim(max(stages) + 5, min(stages) - 5)
    plt.ylim(-5, 105)
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12, loc='best')
    
    for i, txt in enumerate(mcts_rates):
        plt.annotate(f"{txt:.0f}%", (stages[i], mcts_rates[i] + 3), fontsize=12, color='#ff7f0e', ha='center')
    for i, txt in enumerate(minimax_rates):
        plt.annotate(f"{txt:.0f}%", (stages[i], minimax_rates[i] - 6), fontsize=12, color='#1f77b4', ha='center')
        
    plt.tight_layout()
    plt.savefig('MCTS_vs_Minimax_8.png', dpi=300, bbox_inches='tight')
    plt.show()


if __name__ == "__main__":
    stages, mcts_rates, minimax_rates = run_comprehensive_benchmark(time_limit=4.5, num_games_per_stage=40)
    
    plot_results(stages, mcts_rates, minimax_rates)

 Start test(have 10 stages to be tested, each stages has 40 games) 
 Testing phase: There are still 15 empty positions remaining.
The game 1/40 ends | Winner: Minimax
The game 2/40 ends | Winner: Minimax
The game 3/40 ends | Winner: Minimax
The game 4/40 ends | Winner: Minimax
The game 5/40 ends | Winner: MCTS
The game 6/40 ends | Winner: Minimax
The game 7/40 ends | Winner: Minimax
The game 8/40 ends | Winner: MCTS
The game 9/40 ends | Winner: MCTS
The game 10/40 ends | Winner: Minimax
The game 11/40 ends | Winner: Minimax
The game 12/40 ends | Winner: MCTS
The game 13/40 ends | Winner: Minimax
The game 14/40 ends | Winner: Minimax
The game 15/40 ends | Winner: Minimax
The game 16/40 ends | Winner: Minimax
The game 17/40 ends | Winner: Minimax
The game 18/40 ends | Winner: Minimax
The game 19/40 ends | Winner: Minimax
The game 20/40 ends | Winner: MCTS
The game 21/40 ends | Winner: Minimax
The game 22/40 ends | Winner: MCTS
The game 23/40 ends | Winner: Minimax
The game 24/40 ends | W

In [1]:
#cnn 数据测试
import numpy as np
# 导入你的游戏和改好的 MCTS
from OthelloGame import OthelloGame
from OthelloPlayers import PureMCTSPlayer

def execute_episode(game, mcts_player):
    """
    执行一局完整的左右互搏，收集 (s, pi, z) 训练数据。
    """
    train_examples = []
    board = game.getInitBoard()
    cur_player = 1
    episode_step = 0
    
    print("thinking ...")
    
    while True:
        episode_step += 1
        # 1. 始终使用“当前玩家视角”的棋盘
        canonical_board = game.getCanonicalForm(board, cur_player)
        
        # 2. 让 MCTS 思考，获得动作概率分布 pi
        pi = mcts_player.getAction(canonical_board)
        
        # 【记录数据】：存下当前的 盘面、概率、以及是谁在下棋
        train_examples.append([canonical_board, cur_player, pi, None])
        
        # 3. 根据概率 pi 随机抽样选择一个动作去下
        # 这保证了 AI 不会每局都下出完全一模一样的棋
        action = np.random.choice(len(pi), p=pi)
        
        # 4. 执行动作，进入下一个状态
        board, cur_player = game.getNextState(board, cur_player, action)
        
        # 5. 判断游戏是否结束
        r = game.getGameEnded(board, cur_player)
        if r != 0:
            # 游戏结束了！开始给之前记录的数据打上最终胜负标签 z
            print(f"🏁 互搏结束，共进行了 {episode_step} 步！")
            
            # 回溯打标签：如果最后的赢家和下这步棋的人是同一个，z就是赢(+1/胜负值)，否则就是输(-1/胜负值)
            # 因为 r 是相对于游戏结束时的 cur_player 而言的，所以要做一次判断
            for step_idx in range(len(train_examples)):
                is_same_player = (train_examples[step_idx][1] == cur_player)
                # 存入最终的 (s, pi, z)
                train_examples[step_idx][3] = r if is_same_player else -r
                
            # 返回干净的 (s, pi, z) 数据集
            return [(x[0], x[2], x[3]) for x in train_examples]

if __name__ == "__main__":
    game = OthelloGame(8)
    # 为了快速生成数据，我们把限时缩短到 0.5 秒
    mcts_generator = PureMCTSPlayer(game, time_limit=0.5)
    
    # 跑一局互搏
    dataset = execute_episode(game, mcts_generator)
    
    print("\n" + "="*50)
    print("🎉 恭喜！你成功通过自我对弈凭空生成了数据！")
    print(f"这局棋总共产生了 {len(dataset)} 条训练数据。")
    print("="*50)
    
    # 抽查第一步的数据
    s, pi, z = dataset[0]
    
    print("\n【开盲盒：第一条数据样本】")
    print("1. 盘面状态 s (State，即输入给 CNN 的特征):")
    print(s)
    
    print(f"\n2. 策略目标 pi (Policy，即 CNN 第一个头要学习的概率分布，长度 {len(pi)}):")
    # 只打印概率大于 0 的合法动作
    for act, prob in enumerate(pi):
        if prob > 0:
            row, col = int(act / 8), act % 8
            print(f"  - 动作 [{row},{col}] (一维索引 {act}): 概率 {prob*100:.1f}%")
            
    print(f"\n3. 价值目标 z (Value，即 CNN 第二个头要预测的最终结果):")
    print(f"  - 最终结果为: {z} (1代表赢，-1代表输，0代表平)")

thinking ...
🏁 互搏结束，共进行了 61 步！

🎉 恭喜！你成功通过自我对弈凭空生成了数据！
这局棋总共产生了 61 条训练数据。

【开盲盒：第一条数据样本】
1. 盘面状态 s (State，即输入给 CNN 的特征):
[[ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0 -1  1  0  0  0]
 [ 0  0  0  1 -1  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

2. 策略目标 pi (Policy，即 CNN 第一个头要学习的概率分布，长度 65):
  - 动作 [2,3] (一维索引 19): 概率 8.6%
  - 动作 [3,2] (一维索引 26): 概率 54.3%
  - 动作 [4,5] (一维索引 37): 概率 29.6%
  - 动作 [5,4] (一维索引 44): 概率 7.4%

3. 价值目标 z (Value，即 CNN 第二个头要预测的最终结果):
  - 最终结果为: 1 (1代表赢，-1代表输，0代表平)


In [3]:
import sys
import os
import time

sys.path.append(os.getcwd()) 

from OthelloGame import OthelloGame
# 请确保你的 OthelloPlayers.py 中，PureMCTSPlayer 已经替换为了带有 Roxanne 启发式的最新版本！
from OthelloPlayers import RandomPlayer, MinimaxPlayer, PureMCTSPlayer

def quick_literature_test(time_limit=3, num_games=50, test_stage=30):
    """
    极速验证脚本：针对 45 步空位的中盘焦点战，验证 Roxanne 启发式 MCTS 的威力。
    """
    game = OthelloGame(8)
    
    mcts_player = PureMCTSPlayer(game, time_limit=time_limit).play
    minimax_player = MinimaxPlayer(game, time_limit=time_limit).play
    
    print(f"leave {test_stage} empty space")
    print(f"Thinking limited: {time_limit} seconds | Total number of cases: {num_games}")
    print("="*60)
    
    mcts_wins = 0
    minimax_wins = 0
    draws = 0
    
    for i in range(num_games):
        board = game.getInitBoard()
        cur_player = 1
        random_brain = RandomPlayer(game).play
        
        # 制造随机残局：让双方随机下棋，直到剩下 test_stage 个空位
        steps_to_play = 60 - test_stage
        for _ in range(steps_to_play):
            if game.getGameEnded(board, cur_player) != 0:
                break 
            action = random_brain(board) if cur_player == 1 else random_brain(game.getCanonicalForm(board, cur_player))
            board, cur_player = game.getNextState(board, cur_player, action)
            
        # 交替先手，保证比赛公平
        if i % 2 == 0:
            p1, p2 = mcts_player, minimax_player
            p1_name, p2_name = "Roxanne MCTS", "Minimax"
        else:
            p1, p2 = minimax_player, mcts_player
            p1_name, p2_name = "Minimax", "Roxanne MCTS"
            
        # 算法接管比赛
        while game.getGameEnded(board, cur_player) == 0:
            if cur_player == 1:
                action = p1(board)
            else:
                action = p2(game.getCanonicalForm(board, cur_player))
            board, cur_player = game.getNextState(board, cur_player, action)
            
        result = cur_player * game.getGameEnded(board, cur_player)
        winner = p1_name if (result == 1 and cur_player == 1) or (result == -1 and cur_player == -1) else p2_name
        if result == 0:
            winner = "Draw"
            
        if winner == "Roxanne MCTS":
            mcts_wins += 1
        elif winner == "Minimax":
            minimax_wins += 1
        else:
            draws += 1
            
        print(f"局 {i+1}/{num_games} finished -> winner: {winner}")
        
    print("="*60)
    print(f"Roxanne MCTS: {mcts_wins} win | Minimax: {minimax_wins} win | draw: {draws}")
    
    mcts_rate = (mcts_wins / num_games) * 100
    print(f"MCTS wining rate: {mcts_rate:.1f}%")

if __name__ == "__main__":
    quick_literature_test(time_limit=3, num_games=50, test_stage=30)

leave 30 empty space
Thinking limited: 3 seconds | Total number of cases: 50
局 1/50 finished -> winner: Minimax
局 2/50 finished -> winner: Roxanne MCTS
局 3/50 finished -> winner: Minimax
局 4/50 finished -> winner: Minimax
局 5/50 finished -> winner: Roxanne MCTS
局 6/50 finished -> winner: Roxanne MCTS
局 7/50 finished -> winner: Roxanne MCTS
局 8/50 finished -> winner: Minimax
局 9/50 finished -> winner: Roxanne MCTS
局 10/50 finished -> winner: Roxanne MCTS
局 11/50 finished -> winner: Minimax
局 12/50 finished -> winner: Roxanne MCTS
局 13/50 finished -> winner: Minimax
局 14/50 finished -> winner: Roxanne MCTS
局 15/50 finished -> winner: Minimax
局 16/50 finished -> winner: Minimax
局 17/50 finished -> winner: Minimax
局 18/50 finished -> winner: Minimax
局 19/50 finished -> winner: Minimax
局 20/50 finished -> winner: Roxanne MCTS
局 21/50 finished -> winner: Minimax
局 22/50 finished -> winner: Roxanne MCTS
局 23/50 finished -> winner: Minimax
局 24/50 finished -> winner: Roxanne MCTS
局 25/50 finis

In [ ]:
import matplotlib.pyplot as plt

def safe_plot_results(stages, mcts_rates, minimax_rates):
    plt.figure(figsize=(10, 6))
    
    plt.plot(stages, mcts_rates, marker='o', linewidth=3, markersize=10, label='Pure MCTS', color='#ff7f0e')
    plt.plot(stages, minimax_rates, marker='s', linewidth=3, markersize=10, label='Minimax (Alpha-Beta)', color='#1f77b4')
    
    plt.title('Performance Comparison: MCTS vs Minimax at Different Game Stages\n(Time Limit: 6s / move)', fontsize=16, pad=15)
    plt.xlabel('Remaining Empty Spots (Game Complexity)', fontsize=14)
    plt.ylabel('Win Rate (%)', fontsize=14)
    
    plt.xlim(max(stages) + 5, min(stages) - 5)
    plt.ylim(-5, 105)
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12, loc='best')
    
    for i, txt in enumerate(mcts_rates):
        plt.annotate(f"{txt:.0f}%", (stages[i], mcts_rates[i] + 3), fontsize=12, color='#ff7f0e', ha='center')
    for i, txt in enumerate(minimax_rates):
        plt.annotate(f"{txt:.0f}%", (stages[i], minimax_rates[i] - 6), fontsize=12, color='#1f77b4', ha='center')
        
    plt.tight_layout()
    plt.savefig('MCTS_vs_Minimax.png', dpi=300, bbox_inches='tight')
    plt.show()

safe_plot_results(stages, mcts_rates, minimax_rates)

In [ ]:
import sys
import os
import numpy as np

sys.path.append(os.getcwd()) 

from OthelloGame import OthelloGame
from OthelloPlayers import RandomPlayer, GreedyOthelloPlayer, HumanPlayer, MinimaxPlayer, MCTSNode

class Arena():

    def __init__(self, player1_func, player2_func, game, p1_name="Player 1", p2_name="Player 2"):
        self.player1 = player1_func
        self.player2 = player2_func
        self.game = game
        self.p1_name = p1_name
        self.p2_name = p2_name

    def play_match(self, display=False):
        """A single match will be played"""
        board = self.game.getInitBoard()
        curPlayer = 1
        
        step = 0
        while self.game.getGameEnded(board, curPlayer) == 0:
            step += 1
            if display:
                print(f"\n--- 第 {step} 步 --- 当前轮到: {'黑方(1)' if curPlayer==1 else '白方(-1)'}")
                print(board)
                
            if curPlayer == 1:
                action = self.player1(board)
            else:
                canonical_board = self.game.getCanonicalForm(board, curPlayer)
                action = self.player2(canonical_board)
                
            board, curPlayer = self.game.getNextState(board, curPlayer, action)
            
        if display:
            print("\n=== 最终棋盘 ===")
            print(board)
            
        return curPlayer * self.game.getGameEnded(board, curPlayer)

    def play_games(self, num_games, verbose=False):
        """
        进行多局测试，并为了公平起见，半程交换黑白先手。
        """
        p1_wins = 0
        p2_wins = 0
        draws = 0
        
        half_games = num_games // 2

        print(f"========== 评测开始: {self.p1_name} vs {self.p2_name} ({num_games} 局) ==========")
        
        # 前半程：Player 1 执黑先手
        print(f"\n[前半程] {self.p1_name} 执黑(先手) vs {self.p2_name} 执白")
        for i in range(half_games):
            result = self.play_match(display=verbose)
            if result == 1:
                p1_wins += 1
            elif result == -1:
                p2_wins += 1
            else:
                draws += 1
            print(f"局 {i+1}/{half_games} 结束 -> 胜者: {self.p1_name if result==1 else self.p2_name if result==-1 else '平局'}")

        # 后半程：黑白互换 (重要：解决黑白棋先手劣势/优势问题)
        # 注意：在此处，我们在内部调换了 self.player1 和 self.player2 的位置
        self.player1, self.player2 = self.player2, self.player1
        
        print(f"\n[后半程] {self.p2_name} 执黑(先手) vs {self.p1_name} 执白")
        for i in range(half_games):
            result = self.play_match(display=verbose)
            # 因为 player 位置换了，所以 result == 1 代表现在的 player1(即原来的p2) 赢了
            if result == 1:
                p2_wins += 1
            elif result == -1:
                p1_wins += 1
            else:
                draws += 1
            print(f"局 {i+1+half_games}/{num_games} 结束 -> 胜者: {self.p2_name if result==1 else self.p1_name if result==-1 else '平局'}")

        # 恢复原来的玩家顺序
        self.player1, self.player2 = self.player2, self.player1

        print("\n================== 最终评测结果 ==================")
        print(f"{self.p1_name} 获胜: {p1_wins} 局")
        print(f"{self.p2_name} 获胜: {p2_wins} 局")
        print(f"平局: {draws} 局")
        print("==================================================\n")
        
        return p1_wins, p2_wins, draws


# ==========================================
# 实际调用示例
# ==========================================
if __name__ == "__main__":
    game = OthelloGame(8)
    
    # 实例化你想要测试的大脑
    random_brain = RandomPlayer(game).play
    greedy_brain = GreedyOthelloPlayer(game).play
    # Minimax 层数设为 3 或 4 即可，层数太高在纯 Python 环境下会非常慢
    minimax_brain = MinimaxPlayer(game, search_depth=5).play 

    # ----------------------------------------
    # 测试 1: Random vs Greedy
    # ----------------------------------------
    arena1 = Arena(random_brain, greedy_brain, game, p1_name="Random_AI", p2_name="Greedy_AI")
    # arena1.play_games(10, verbose=False) # 取消注释即可运行

    # ----------------------------------------
    # 测试 2: Greedy vs Minimax (核心验证)
    # ----------------------------------------
    arena2 = Arena(greedy_brain, minimax_brain, game, p1_name="Greedy_AI", p2_name="Minimax_AI(Depth=3)")
    arena2.play_games(6, verbose=False)

In [ ]:
import sys
import os
import numpy as np

sys.path.append(os.getcwd()) 

from OthelloGame import OthelloGame
from OthelloPlayers import RandomPlayer, GreedyOthelloPlayer, HumanPlayer, MinimaxPlayer, PureMCTSPlayer

class Arena():

    def __init__(self, player1_func, player2_func, game, p1_name="Player 1", p2_name="Player 2"):
        self.player1 = player1_func
        self.player2 = player2_func
        self.game = game
        self.p1_name = p1_name
        self.p2_name = p2_name

    def play_match(self, display=False):
        """A single match will be played"""
        board = self.game.getInitBoard()
        curPlayer = 1
        
        step = 0
        while self.game.getGameEnded(board, curPlayer) == 0:
            step += 1
            if display:
                print(f"\n--- Step {step} --- Current turn: {'Black (1)' if curPlayer==1 else 'White (-1)'}")
                print(board)
                
            if curPlayer == 1:
                action = self.player1(board)
            else:
                canonical_board = self.game.getCanonicalForm(board, curPlayer)
                action = self.player2(canonical_board)
                
            board, curPlayer = self.game.getNextState(board, curPlayer, action)
            
        if display:
            print("\n=== Final Board ===")
            print(board)
            
        return curPlayer * self.game.getGameEnded(board, curPlayer)

    def play_games(self, num_games, verbose=False):
        """
        Play multiple test games, swapping black/white colors at the halfway point for fairness.
        """
        p1_wins = 0
        p2_wins = 0
        draws = 0
        
        half_games = num_games // 2

        print(f"========== Evaluation Started: {self.p1_name} vs {self.p2_name} ({num_games} games) ==========")
        
        # First half: Player 1 plays Black (first to move)
        print(f"\n[First Half] {self.p1_name} plays Black (First) vs {self.p2_name} plays White")
        for i in range(half_games):
            result = self.play_match(display=verbose)
            if result == 1:
                p1_wins += 1
            elif result == -1:
                p2_wins += 1
            else:
                draws += 1
            print(f"Game {i+1}/{half_games} Ended -> Winner: {self.p1_name if result==1 else self.p2_name if result==-1 else 'Draw'}")

        # Second half: Swap Black and White (Important: balances first-mover advantage/disadvantage in Othello)
        # Note: Here we internally swap the positions of self.player1 and self.player2
        self.player1, self.player2 = self.player2, self.player1
        
        print(f"\n[Second Half] {self.p2_name} plays Black (First) vs {self.p1_name} plays White")
        for i in range(half_games):
            result = self.play_match(display=verbose)
            # Because player positions swapped, result == 1 means the current player1 (original p2) won
            if result == 1:
                p2_wins += 1
            elif result == -1:
                p1_wins += 1
            else:
                draws += 1
            print(f"Game {i+1+half_games}/{num_games} Ended -> Winner: {self.p2_name if result==1 else self.p1_name if result==-1 else 'Draw'}")

        # Restore the original player order
        self.player1, self.player2 = self.player2, self.player1

        print("\n================== Final Evaluation Results ==================")
        print(f"{self.p1_name} Wins: {p1_wins} games")
        print(f"{self.p2_name} Wins: {p2_wins} games")
        print(f"Draws: {draws} games")
        print("==================================================\n")
        
        return p1_wins, p2_wins, draws


# Practical Usage Example

if __name__ == "__main__":
    game = OthelloGame(8)
    
    # Instantiate the brains you want to test
    random_brain = RandomPlayer(game).play
    greedy_brain = GreedyOthelloPlayer(game).play
    
    mcts_brain = PureMCTSPlayer(game, num_simulations=100).play
    minimax_brain = MinimaxPlayer(game, search_depth=4).play 

    # ----------------------------------------
    # Setup Arenas
    # ----------------------------------------
    # TEST1: Baseline(Random VS Gready)
    arena_rand_vs_greedy = Arena(random_brain, greedy_brain, game, p1_name="Random_AI", p2_name="Greedy_AI")
    
    # TEST2 : Greedy VS Minmax(4)
    arena_greedy_vs_mini = Arena(greedy_brain, minimax_brain, game, p1_name="Greedy_AI", p2_name="Minimax_AI(Depth=4)")
    
    # TEST3: Greedy VS MCTS(100)
    arena_greedy_vs_mcts = Arena(greedy_brain, mcts_brain, game, p1_name="Greedy_AI", p2_name="Pure_MCTS(n=100)")
    
    # TEST4: Minmax(4) VS MCTS(100)
    arena_mcts_vs_mini = Arena(mcts_brain, minimax_brain, game, p1_name="Pure_MCTS(n=100)", p2_name="Minimax_AI(Depth=4)")


    print("READY TO START...\n")
    
    # arena_rand_vs_greedy.play_games(10, verbose=False)
    
    # arena_greedy_vs_mini.play_games(6, verbose=False)
    
    # arena_greedy_vs_mcts.play_games(6, verbose=False)
    
    # 当前激活的是测试 4 (MCTS vs Minimax)
    arena_mcts_vs_mini.play_games(6, verbose=False)

In [ ]:
import sys
import os
sys.path.append(os.getcwd()) 

from OthelloGame import OthelloGame
from OthelloPlayers import HumanPlayer, RandomPlayer, GreedyOthelloPlayer, MinimaxPlayer

game = OthelloGame(8)

human_brain = HumanPlayer(game).play

# Choose opponent
# ai_brain = RandomPlayer(game).play                # Difficulty: Baby
# ai_brain = GreedyOthelloPlayer(game).play         # Difficulty: Beginner
ai_brain = MinimaxPlayer(game, search_depth=4).play # Difficulty: Expert (Recommended depth 4-5)

# Determine play order
# If you want to play Black (First move): player1 = human_brain, player2 = ai_brain
# If you want to play White (Second move): player1 = ai_brain, player2 = human_brain
p1 = human_brain
p2 = ai_brain

p1_name = "Human Player (Black)"
p2_name = "Minimax_AI (White)"

print(f"========== The Century Human vs AI Match ==========")
print(f"{p1_name} vs {p2_name}")
print("Hint: When it's your turn, Jupyter will pop up an input box.")
print("Input format is 'row col' (e.g., '3 4'). Note that coordinates start from 0!")
print("===================================================\n")


# Start a Single Game
# Instantiate the Arena
human_vs_ai_arena = Arena(p1, p2, game, p1_name=p1_name, p2_name=p2_name)
result = human_vs_ai_arena.play_match(display=True)


print("\n================ Final Result ================")
if result == 1:
    print(f" Congratulations! [{p1_name}] has won the game!")
elif result == -1:
    print(f" Unfortunately, [{p2_name}] has won the game!")
else:
    print(" A close match, it's a draw!")